# Gymnasium Quickstart with PPO

This notebook provides a minimal but complete introduction to Gymnasium and implements PPO (Proximal Policy Optimization) with easy-to-use visualization.

## Contents
1. Gymnasium Basics
2. Environment Exploration
3. Random Agent Baseline
4. PPO Training
5. Visualization & Testing

In [ ]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, display
import imageio
from stable_baselines3 import PPO
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.callbacks import BaseCallback
from tqdm.notebook import tqdm
import warnings
warnings.filterwarnings('ignore')

print("✓ All imports successful!")
print(f"Gymnasium version: {gym.__version__}")

## 1. Gymnasium Basics

Gymnasium is the maintained fork of OpenAI Gym. It provides a simple interface for RL environments.

In [ ]:
# Create an environment
env = gym.make('CartPole-v1', render_mode='rgb_array')

# Inspect the environment
print("Environment: CartPole-v1")
print(f"Observation space: {env.observation_space}")
print(f"Action space: {env.action_space}")
print(f"\nObservation space shape: {env.observation_space.shape}")
print(f"Number of actions: {env.action_space.n}")
print(f"\nMax episode steps: {env.spec.max_episode_steps}")

## 2. Environment Exploration

Let's explore what the observations and actions mean:

In [ ]:
# Reset the environment and get initial observation
observation, info = env.reset(seed=42)

print("Initial observation:", observation)
print("\nObservation components:")
print(f"  [0] Cart position: {observation[0]:.3f}")
print(f"  [1] Cart velocity: {observation[1]:.3f}")
print(f"  [2] Pole angle: {observation[2]:.3f} rad")
print(f"  [3] Pole angular velocity: {observation[3]:.3f} rad/s")
print("\nActions:")
print("  0: Push cart to the left")
print("  1: Push cart to the right")

## 3. Visualization Helper Functions

These functions make it easy to visualize agent behavior:

In [ ]:
def render_episode(env, agent=None, max_steps=500, seed=None):
    """
    Render a single episode and return frames.
    
    Args:
        env: Gymnasium environment
        agent: Trained agent (if None, uses random actions)
        max_steps: Maximum steps per episode
        seed: Random seed for reproducibility
    
    Returns:
        frames: List of RGB arrays
        total_reward: Cumulative reward
        steps: Number of steps taken
    """
    frames = []
    obs, _ = env.reset(seed=seed)
    total_reward = 0
    
    for step in range(max_steps):
        # Render and store frame
        frame = env.render()
        frames.append(frame)
        
        # Get action
        if agent is None:
            action = env.action_space.sample()  # Random action
        else:
            action, _ = agent.predict(obs, deterministic=True)
        
        # Step environment
        obs, reward, terminated, truncated, _ = env.step(action)
        total_reward += reward
        
        if terminated or truncated:
            break
    
    return frames, total_reward, step + 1


def display_frames(frames, title="Episode", figsize=(8, 6)):
    """
    Display frames as an animated matplotlib figure.
    """
    fig, ax = plt.subplots(figsize=figsize)
    ax.axis('off')
    ax.set_title(title, fontsize=14, fontweight='bold')
    
    img = ax.imshow(frames[0])
    
    def animate(i):
        img.set_array(frames[i])
        return [img]
    
    anim = animation.FuncAnimation(
        fig, animate, frames=len(frames), 
        interval=50, blit=True, repeat=True
    )
    
    plt.close()  # Don't show static plot
    return HTML(anim.to_jshtml())


def save_video(frames, filename='episode.mp4', fps=30):
    """
    Save frames as a video file.
    """
    imageio.mimsave(filename, frames, fps=fps)
    print(f"Video saved to {filename}")


def compare_agents(env, agents_dict, num_episodes=5, seed=42):
    """
    Compare multiple agents side by side.
    
    Args:
        env: Gymnasium environment
        agents_dict: Dictionary of {name: agent} (use None for random)
        num_episodes: Number of episodes to average
        seed: Random seed
    """
    results = {}
    
    for name, agent in agents_dict.items():
        rewards = []
        steps = []
        
        for ep in range(num_episodes):
            _, reward, step = render_episode(env, agent, seed=seed+ep)
            rewards.append(reward)
            steps.append(step)
        
        results[name] = {
            'mean_reward': np.mean(rewards),
            'std_reward': np.std(rewards),
            'mean_steps': np.mean(steps),
            'std_steps': np.std(steps)
        }
    
    # Display results
    print("\n" + "="*60)
    print("AGENT COMPARISON RESULTS")
    print("="*60)
    for name, stats in results.items():
        print(f"\n{name}:")
        print(f"  Mean Reward: {stats['mean_reward']:.2f} ± {stats['std_reward']:.2f}")
        print(f"  Mean Steps:  {stats['mean_steps']:.1f} ± {stats['std_steps']:.1f}")
    print("\n" + "="*60)
    
    return results

print("✓ Visualization functions loaded!")

## 4. Random Agent Baseline

Let's see how a random agent performs:

In [ ]:
# Visualize random agent
frames, reward, steps = render_episode(env, agent=None, seed=42)
print(f"Random Agent - Reward: {reward:.1f}, Steps: {steps}")

# Display animation
display(display_frames(frames, title="Random Agent"))

## 5. Training Progress Callback

This callback shows real-time training progress:

In [ ]:
class ProgressCallback(BaseCallback):
    """
    Custom callback for displaying training progress.
    """
    def __init__(self, check_freq=1000, verbose=1):
        super().__init__(verbose)
        self.check_freq = check_freq
        self.episode_rewards = []
        self.episode_lengths = []
        self.current_rewards = []
        self.current_reward = 0
        
    def _on_step(self):
        # Track reward
        self.current_reward += self.locals['rewards'][0]
        
        # Check if episode ended
        if self.locals['dones'][0]:
            self.episode_rewards.append(self.current_reward)
            self.current_reward = 0
        
        # Periodic logging
        if self.n_calls % self.check_freq == 0:
            if len(self.episode_rewards) > 0:
                recent_rewards = self.episode_rewards[-100:]
                mean_reward = np.mean(recent_rewards)
                print(f"Steps: {self.num_timesteps:6d} | "
                      f"Episodes: {len(self.episode_rewards):4d} | "
                      f"Mean Reward (last 100): {mean_reward:.2f}")
        
        return True

print("✓ Training callback loaded!")

## 6. PPO Training

Now let's train a PPO agent. PPO is one of the most popular RL algorithms:

- **Stable**: Uses clipped objective to prevent large policy updates
- **Sample efficient**: Reuses experience multiple times
- **Easy to tune**: Few hyperparameters, robust defaults

In [ ]:
# Create training environment
train_env = gym.make('CartPole-v1')

# Create PPO agent
print("Creating PPO agent...")
model = PPO(
    'MlpPolicy',           # Multi-layer perceptron policy
    train_env,
    learning_rate=3e-4,    # Learning rate
    n_steps=2048,          # Steps per rollout
    batch_size=64,         # Minibatch size
    n_epochs=10,           # Optimization epochs per rollout
    gamma=0.99,            # Discount factor
    gae_lambda=0.95,       # GAE lambda for advantage estimation
    clip_range=0.2,        # PPO clipping parameter
    verbose=0
)

print("✓ PPO agent created!")
print(f"\nPolicy architecture:")
print(model.policy)

In [ ]:
# Train the agent
print("\nTraining PPO agent...")
print("="*60)

callback = ProgressCallback(check_freq=2000)

# Training loop
model.learn(
    total_timesteps=50000,  # Adjust for longer/shorter training
    callback=callback,
    progress_bar=True
)

print("\n" + "="*60)
print("✓ Training complete!")

# Save the model
model.save("ppo_cartpole")
print("✓ Model saved to 'ppo_cartpole.zip'")

## 7. Training Progress Visualization

In [ ]:
# Plot training progress
rewards = callback.episode_rewards

# Calculate moving average
window = 50
moving_avg = np.convolve(rewards, np.ones(window)/window, mode='valid')

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(rewards, alpha=0.3, label='Episode Reward')
plt.plot(range(window-1, len(rewards)), moving_avg, 
         linewidth=2, label=f'{window}-Episode Moving Average')
plt.xlabel('Episode')
plt.ylabel('Reward')
plt.title('Training Progress')
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.hist(rewards[-500:], bins=30, edgecolor='black', alpha=0.7)
plt.xlabel('Reward')
plt.ylabel('Frequency')
plt.title('Reward Distribution (Last 500 Episodes)')
plt.axvline(np.mean(rewards[-500:]), color='red', linestyle='--', 
            linewidth=2, label=f'Mean: {np.mean(rewards[-500:]):.1f}')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nFinal Performance (last 500 episodes):")
print(f"  Mean reward: {np.mean(rewards[-500:]):.2f}")
print(f"  Std reward: {np.std(rewards[-500:]):.2f}")
print(f"  Max reward: {np.max(rewards[-500:]):.2f}")

## 8. Evaluate Trained Agent

In [ ]:
# Evaluate the trained agent
eval_env = gym.make('CartPole-v1', render_mode='rgb_array')

mean_reward, std_reward = evaluate_policy(
    model, 
    eval_env, 
    n_eval_episodes=20,
    deterministic=True
)

print(f"\nEvaluation Results (20 episodes):")
print(f"  Mean reward: {mean_reward:.2f} ± {std_reward:.2f}")
print(f"  Target: 500.0 (max possible)")

if mean_reward >= 475:
    print("  🎉 Excellent! Agent has mastered the task!")
elif mean_reward >= 400:
    print("  ✓ Great! Agent performs well!")
elif mean_reward >= 200:
    print("  → Good progress, but could improve with more training")
else:
    print("  ⚠ Agent needs more training")

## 9. Visualize Trained Agent

Let's see the trained agent in action!

In [ ]:
# Render trained agent
frames, reward, steps = render_episode(eval_env, agent=model, seed=42)
print(f"Trained PPO Agent - Reward: {reward:.1f}, Steps: {steps}")

# Display animation
display(display_frames(frames, title="Trained PPO Agent"))

## 10. Side-by-Side Comparison

In [ ]:
# Compare random vs trained agent
comparison = compare_agents(
    eval_env,
    {
        'Random Agent': None,
        'Trained PPO': model
    },
    num_episodes=10,
    seed=42
)

# Visualize comparison
names = list(comparison.keys())
rewards = [comparison[name]['mean_reward'] for name in names]
errors = [comparison[name]['std_reward'] for name in names]

plt.figure(figsize=(10, 6))
bars = plt.bar(names, rewards, yerr=errors, capsize=10, 
               color=['#ff7f0e', '#2ca02c'], alpha=0.8, edgecolor='black')
plt.ylabel('Mean Reward', fontsize=12)
plt.title('Agent Performance Comparison', fontsize=14, fontweight='bold')
plt.ylim(0, max(rewards) * 1.2)
plt.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for bar, reward in zip(bars, rewards):
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
             f'{reward:.1f}',
             ha='center', va='bottom', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

## 11. Save Video (Optional)

Save the trained agent's performance as a video file:

In [ ]:
# Generate and save video
frames, reward, steps = render_episode(eval_env, agent=model, seed=123)
save_video(frames, 'ppo_cartpole_demo.mp4', fps=30)
print(f"Episode: Reward={reward:.1f}, Steps={steps}")

## 12. Load Saved Model

You can load the saved model later:

In [ ]:
# Load the saved model
loaded_model = PPO.load("ppo_cartpole")
print("✓ Model loaded successfully!")

# Test loaded model
frames, reward, steps = render_episode(eval_env, agent=loaded_model, seed=999)
print(f"Loaded Model - Reward: {reward:.1f}, Steps: {steps}")

display(display_frames(frames, title="Loaded Model Test"))

## 13. Try Different Environments

The same approach works with other Gymnasium environments:

In [ ]:
# List of simple environments to try
simple_envs = [
    'Acrobot-v1',
    'MountainCar-v0', 
    'LunarLander-v3',
    'Pendulum-v1'
]

print("Other environments you can try:")
for env_name in simple_envs:
    print(f"  • {env_name}")

print("\nTo train on a different environment:")
print("  1. Change 'CartPole-v1' to your chosen environment")
print("  2. You may need to adjust hyperparameters")
print("  3. Increase timesteps for harder environments")

## Summary

### What we learned:

1. **Gymnasium basics**: How to create environments, inspect observation/action spaces
2. **Random baseline**: Always establish a baseline for comparison
3. **PPO training**: How to train a state-of-the-art RL algorithm
4. **Visualization**: Multiple ways to observe and compare agent behavior
5. **Model persistence**: How to save and load trained agents

### Key takeaways:

- **Start simple**: CartPole is perfect for learning and debugging
- **Visualize often**: Watching your agent helps debug issues
- **Compare agents**: Always compare to baselines (random, previous versions)
- **Save models**: Don't lose your trained agents!
- **Monitor training**: Use callbacks to track progress

### Next steps:

1. Try different environments (Acrobot, LunarLander, etc.)
2. Experiment with hyperparameters
3. Try other algorithms (A2C, SAC, TD3)
4. Build custom environments
5. Implement curriculum learning

In [ ]:
# Cleanup
env.close()
train_env.close()
eval_env.close()
print("✓ Environments closed.")